# MegaMart — Gold Layer & Power BI Data Model

**Pipeline:** Bronze/Raw → Silver → Gold → Power BI

This notebook creates persistent Gold tables from the existing Silver tables. The project documentation is the primary source of truth.


## 1. Project Objective

Create a reusable Gold layer supporting the documented Sales Performance, Inventory Optimization, and Comparative Store Performance requirements.

The documented project period is Jan–Dec 2024 and the chain has five stores. Gold uses the cleaned Silver layer and does not repeat Silver-layer cleaning.


## 2. Gold-Layer Architecture

### Dimensions
- `gold_dim_date`
- `gold_dim_store`
- `gold_dim_product`
- `gold_dim_supplier`

### Facts
- `gold_fact_sales`
- `gold_fact_inventory`
- `gold_fact_purchase_orders`

### Analytical marts
- `gold_store_performance`
- `gold_store_sales_monthly`
- `gold_product_performance`
- `gold_inventory_analysis`
- `gold_supplier_performance`
- `gold_sales_monthly`

These are persistent, reusable tables rather than one table per question.


## 3. Data-Quality and Business Rules

The Silver layer is assumed to have already handled the documented duplicate, missing-value, invalid-price, invalid-date, orphan-key, phone-format, quantity, and delivery-date issues.

Gold rules:
1. Sales are restricted to 2024.
2. Revenue = `quantity_sold * unit_price`, represented by Silver `total_amount`.
3. Transaction count = distinct `transaction_id`.
4. Low stock = `< 50` units.
5. Zero stock = out of stock.
6. No profit metric is created because no valid product cost field is available.
7. True supplier on-time delivery rate is not calculated because no promised/expected delivery date exists.
8. Formal inventory turnover is not claimed because cost and historical average inventory are unavailable. A clearly named operational proxy is provided.
9. Formal reorder point normally requires demand during lead time plus safety stock. Since safety-stock/service-level inputs are absent, a reorder-point proxy is provided.
10. Slow-moving/overstock thresholds are explicit assumptions and are documented in the inventory section.


In [0]:
%sql
SHOW TABLES LIKE '*_silver';

## 4. Dimension Tables

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_date AS
WITH dates AS (
    SELECT explode(sequence(to_date('2024-01-01'), to_date('2024-12-31'), interval 1 day)) AS date
)
SELECT
    date,
    year(date) AS year,
    quarter(date) AS quarter,
    month(date) AS month_number,
    date_format(date, 'MMMM') AS month_name,
    date_format(date, 'yyyy-MM') AS year_month,
    dayofmonth(date) AS day_of_month,
    dayofweek(date) AS day_of_week,
    date_format(date, 'EEEE') AS day_name
FROM dates;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_store AS
SELECT store_id, store_name, location, manager_name
FROM stores_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_supplier AS
SELECT supplier_id, supplier_name, contact_person, phone
FROM suppliers_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_product AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.unit_price,
    p.supplier_id,
    s.supplier_name
FROM products_silver p
LEFT JOIN suppliers_silver s ON p.supplier_id = s.supplier_id;

## 5. Fact Tables

Fact grain:
- `gold_fact_sales`: one cleaned transaction.
- `gold_fact_inventory`: one store-product inventory record.
- `gold_fact_purchase_orders`: one purchase order.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_sales AS
SELECT
    s.transaction_id,
    CAST(s.sale_date AS DATE) AS sale_date,
    s.store_id,
    s.product_id,
    p.supplier_id,
    s.quantity_sold,
    s.unit_price,
    ROUND(s.total_amount, 2) AS total_amount
FROM sales_silver s
INNER JOIN stores_silver st ON s.store_id = st.store_id
INNER JOIN products_silver p ON s.product_id = p.product_id
WHERE s.sale_date BETWEEN '2024-01-01' AND '2024-12-31';

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_inventory AS
SELECT
    i.inventory_id,
    i.store_id,
    i.product_id,
    p.supplier_id,
    i.quantity_on_hand,
    p.unit_price,
    ROUND(i.quantity_on_hand * p.unit_price, 2) AS inventory_retail_value
FROM inventory_silver i
INNER JOIN stores_silver st ON i.store_id = st.store_id
INNER JOIN products_silver p ON i.product_id = p.product_id
WHERE i.quantity_on_hand >= 0;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_purchase_orders AS
SELECT
    po.po_id,
    po.supplier_id,
    CAST(po.order_date AS DATE) AS order_date,
    CAST(po.delivery_date AS DATE) AS delivery_date,
    CASE WHEN po.delivery_date IS NOT NULL
         THEN datediff(po.delivery_date, po.order_date)
         ELSE NULL END AS delivery_days,
    CASE WHEN po.delivery_date IS NULL
         THEN 'Pending/Cancelled'
         ELSE 'Delivered' END AS delivery_status
FROM purchase_orders_silver po;

## 6. Store Performance Gold Table

Supports revenue, average transaction value, unique products, inventory movement, category sales, market share, monthly trend, rankings, and transaction count.

**Inventory turnover caution:** the formal accounting ratio cannot be calculated from the available schema. `inventory_turnover_proxy = 2024 units sold / current units on hand`.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_store_performance AS
WITH sales AS (
    SELECT
        store_id,
        SUM(total_amount) AS total_revenue,
        COUNT(DISTINCT transaction_id) AS transaction_count,
        COUNT(DISTINCT product_id) AS unique_products_sold,
        SUM(quantity_sold) AS units_sold
    FROM gold_fact_sales
    GROUP BY store_id
),
inventory AS (
    SELECT store_id, SUM(quantity_on_hand) AS current_units_on_hand
    FROM gold_fact_inventory
    GROUP BY store_id
),
category_sales AS (
    SELECT
        f.store_id,
        p.category,
        SUM(f.total_amount) AS category_revenue
    FROM gold_fact_sales f
    JOIN gold_dim_product p ON f.product_id = p.product_id
    GROUP BY f.store_id, p.category
),
ranked_categories AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY store_id
            ORDER BY category_revenue DESC, category
        ) AS category_rank
    FROM category_sales
),
top_categories AS (
    SELECT
        store_id,
        concat_ws(', ', sort_array(collect_list(
            CASE WHEN category_rank <= 3
                 THEN concat(category, ' (', cast(round(category_revenue,2) AS STRING), ')')
            END
        ))) AS top_3_categories_by_sales
    FROM ranked_categories
    GROUP BY store_id
),
chain AS (
    SELECT SUM(total_amount) AS chain_revenue
    FROM gold_fact_sales
)
SELECT
    st.store_id,
    st.store_name,
    st.location,
    st.manager_name,
    COALESCE(s.total_revenue,0) AS total_revenue,
    COALESCE(s.transaction_count,0) AS transaction_count,
    ROUND(CASE WHEN COALESCE(s.transaction_count,0) > 0
              THEN s.total_revenue/s.transaction_count END,2) AS average_transaction_value,
    COALESCE(s.unique_products_sold,0) AS unique_products_sold,
    COALESCE(s.units_sold,0) AS units_sold,
    COALESCE(i.current_units_on_hand,0) AS current_units_on_hand,
    ROUND(CASE WHEN COALESCE(i.current_units_on_hand,0) > 0
              THEN s.units_sold/i.current_units_on_hand END,4) AS inventory_turnover_proxy,
    ROUND(CASE WHEN chain.chain_revenue > 0
              THEN s.total_revenue/chain.chain_revenue*100 END,2) AS market_share_percent,
    DENSE_RANK() OVER (ORDER BY COALESCE(s.total_revenue,0) DESC) AS revenue_rank,
    DENSE_RANK() OVER (ORDER BY COALESCE(s.transaction_count,0) DESC) AS transaction_count_rank,
    tc.top_3_categories_by_sales
FROM gold_dim_store st
LEFT JOIN sales s ON st.store_id=s.store_id
LEFT JOIN inventory i ON st.store_id=i.store_id
CROSS JOIN chain
LEFT JOIN top_categories tc ON st.store_id=tc.store_id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_store_sales_monthly AS
WITH monthly AS (
    SELECT
        f.store_id,
        d.year,
        d.month_number,
        d.month_name,
        d.year_month,
        SUM(f.total_amount) AS monthly_revenue,
        SUM(f.quantity_sold) AS monthly_units_sold,
        COUNT(DISTINCT f.transaction_id) AS monthly_transaction_count
    FROM gold_fact_sales f
    JOIN gold_dim_date d ON f.sale_date=d.date
    GROUP BY f.store_id,d.year,d.month_number,d.month_name,d.year_month
)
SELECT
    *,
    LAG(monthly_revenue) OVER (
        PARTITION BY store_id ORDER BY year,month_number
    ) AS previous_month_revenue,
    ROUND(
        CASE WHEN LAG(monthly_revenue) OVER (
            PARTITION BY store_id ORDER BY year,month_number
        ) <> 0
        THEN (monthly_revenue-LAG(monthly_revenue) OVER (
            PARTITION BY store_id ORDER BY year,month_number
        )) / LAG(monthly_revenue) OVER (
            PARTITION BY store_id ORDER BY year,month_number
        ) * 100 END,2
    ) AS month_over_month_growth_percent
FROM monthly;

## 7. Product Performance Gold Table

Supports top products by quantity, revenue/category analysis, product diversity, and sales velocity.

**Profitability limitation:** selling price is available, but product cost is not. Therefore `total_revenue` is provided; gross profit and margin are not calculated.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_product_performance AS
WITH ps AS (
    SELECT
        store_id,
        product_id,
        SUM(quantity_sold) AS total_quantity_sold,
        SUM(total_amount) AS total_revenue,
        COUNT(DISTINCT transaction_id) AS transaction_count,
        MIN(sale_date) AS first_sale_date,
        MAX(sale_date) AS last_sale_date
    FROM gold_fact_sales
    GROUP BY store_id,product_id
)
SELECT
    ps.store_id,
    st.store_name,
    ps.product_id,
    p.product_name,
    p.category,
    p.supplier_id,
    p.supplier_name,
    p.unit_price,
    ps.total_quantity_sold,
    ps.total_revenue,
    ps.transaction_count,
    ps.first_sale_date,
    ps.last_sale_date,
    ROUND(ps.total_quantity_sold/366.0,4) AS average_daily_sales_velocity,
    DENSE_RANK() OVER (
        PARTITION BY ps.store_id ORDER BY ps.total_revenue DESC
    ) AS product_revenue_rank_in_store,
    DENSE_RANK() OVER (
        PARTITION BY ps.store_id ORDER BY ps.total_quantity_sold DESC
    ) AS product_quantity_rank_in_store
FROM ps
JOIN gold_dim_store st ON ps.store_id=st.store_id
JOIN gold_dim_product p ON ps.product_id=p.product_id;

## 8. Inventory Analysis Gold Table

### Explicit assumptions
- `slow_moving_flag`: velocity below the average positive velocity across store-product combinations.
- `overstock_candidate_flag`: slow-moving and current stock greater than 30 days of observed average daily demand.
- `reorder_point_proxy_units`: average daily sales velocity × average supplier delivery days.

The 30-day overstock threshold and reorder formula are analytical proxies, not requirements stated numerically in the documentation.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_inventory_analysis AS
WITH sales AS (
    SELECT store_id,product_id,
           SUM(quantity_sold) AS units_sold_2024,
           SUM(total_amount) AS revenue_2024
    FROM gold_fact_sales
    GROUP BY store_id,product_id
),
lead_time AS (
    SELECT supplier_id, AVG(delivery_days) AS average_delivery_days
    FROM gold_fact_purchase_orders
    WHERE delivery_date IS NOT NULL AND delivery_days >= 0
    GROUP BY supplier_id
),
base AS (
    SELECT
        i.inventory_id,
        i.store_id,
        st.store_name,
        i.product_id,
        p.product_name,
        p.category,
        p.supplier_id,
        p.supplier_name,
        i.quantity_on_hand,
        p.unit_price,
        i.inventory_retail_value,
        COALESCE(s.units_sold_2024,0) AS units_sold_2024,
        COALESCE(s.revenue_2024,0) AS revenue_2024,
        ROUND(COALESCE(s.units_sold_2024,0)/366.0,4) AS average_daily_sales_velocity,
        lt.average_delivery_days
    FROM gold_fact_inventory i
    JOIN gold_dim_store st ON i.store_id=st.store_id
    JOIN gold_dim_product p ON i.product_id=p.product_id
    LEFT JOIN sales s ON i.store_id=s.store_id AND i.product_id=s.product_id
    LEFT JOIN lead_time lt ON p.supplier_id=lt.supplier_id
),
stats AS (
    SELECT AVG(average_daily_sales_velocity) AS avg_positive_velocity
    FROM base
    WHERE average_daily_sales_velocity > 0
)
SELECT
    b.*,
    CASE WHEN b.quantity_on_hand=0 THEN 'Out of Stock'
         WHEN b.quantity_on_hand<50 THEN 'Low Stock'
         ELSE 'In Stock' END AS stock_status,
    CASE WHEN b.average_daily_sales_velocity>0
              AND b.average_daily_sales_velocity<stats.avg_positive_velocity
         THEN TRUE ELSE FALSE END AS slow_moving_flag,
    CASE WHEN b.average_daily_sales_velocity>0
              AND b.average_daily_sales_velocity<stats.avg_positive_velocity
              AND b.quantity_on_hand>b.average_daily_sales_velocity*30
         THEN TRUE ELSE FALSE END AS overstock_candidate_flag,
    ROUND(CASE WHEN b.quantity_on_hand>0
               THEN b.units_sold_2024/b.quantity_on_hand END,4)
         AS inventory_turnover_proxy,
    ROUND(CASE WHEN b.average_delivery_days IS NOT NULL
               THEN b.average_daily_sales_velocity*b.average_delivery_days END,2)
         AS reorder_point_proxy_units,
    CASE
        WHEN b.quantity_on_hand=0 AND b.average_daily_sales_velocity>0
            THEN 'Reorder — stockout with observed demand'
        WHEN b.average_delivery_days IS NOT NULL
             AND b.average_daily_sales_velocity>0
             AND b.quantity_on_hand < b.average_daily_sales_velocity*b.average_delivery_days
            THEN 'Reorder — stock below lead-time demand proxy'
        WHEN b.average_daily_sales_velocity=0 AND b.quantity_on_hand>0
            THEN 'Review — no observed 2024 sales'
        WHEN b.quantity_on_hand<50
            THEN 'Review — low stock'
        WHEN b.average_daily_sales_velocity>0
             AND b.quantity_on_hand>b.average_daily_sales_velocity*30
            THEN 'Review — potential overstock'
        ELSE 'No immediate action'
    END AS procurement_recommendation
FROM base b
CROSS JOIN stats;

## 9. Supplier Performance Gold Table

The source contains order and actual delivery dates, but no promised/expected date. Therefore a valid **on-time delivery rate cannot be calculated**.

Gold provides average delivery days, delivery completion rate, and a clearly named reliability proxy rank. Do not label completion rate as on-time rate.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_supplier_performance AS
SELECT
    s.supplier_id,
    s.supplier_name,
    s.contact_person,
    s.phone,
    COUNT(po.po_id) AS total_purchase_orders,
    SUM(CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN po.delivery_date IS NULL THEN 1 ELSE 0 END) AS pending_or_cancelled_orders,
    ROUND(CASE WHEN COUNT(po.po_id)>0
               THEN SUM(CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END)
                    / COUNT(po.po_id)*100 END,2) AS delivery_completion_rate_percent,
    ROUND(AVG(CASE WHEN po.delivery_date IS NOT NULL THEN po.delivery_days END),2)
        AS average_delivery_days,
    CAST(NULL AS DOUBLE) AS on_time_delivery_rate_percent,
    DENSE_RANK() OVER (
        ORDER BY
            CASE WHEN COUNT(po.po_id)>0
                 THEN SUM(CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END)
                      / COUNT(po.po_id)
                 ELSE 0 END DESC,
            AVG(CASE WHEN po.delivery_date IS NOT NULL THEN po.delivery_days END) ASC
    ) AS supplier_reliability_proxy_rank
FROM gold_dim_supplier s
LEFT JOIN gold_fact_purchase_orders po ON s.supplier_id=po.supplier_id
GROUP BY s.supplier_id,s.supplier_name,s.contact_person,s.phone;

## 10. Chain-Level Monthly Sales

In [0]:
%sql
CREATE OR REPLACE TABLE gold_sales_monthly AS
SELECT
    d.year,
    d.month_number,
    d.month_name,
    d.year_month,
    f.store_id,
    st.store_name,
    SUM(f.quantity_sold) AS units_sold,
    COUNT(DISTINCT f.transaction_id) AS transaction_count,
    SUM(f.total_amount) AS revenue
FROM gold_fact_sales f
JOIN gold_dim_date d ON f.sale_date=d.date
JOIN gold_dim_store st ON f.store_id=st.store_id
GROUP BY d.year,d.month_number,d.month_name,d.year_month,f.store_id,st.store_name;

## 11. Validation / Data-Quality Checks

Run these after the Gold tables are created. Issue queries should return no rows/counts when the corresponding condition is clean.


In [0]:
%sql
-- Duplicate transaction IDs
SELECT transaction_id,COUNT(*) AS row_count
FROM gold_fact_sales
GROUP BY transaction_id
HAVING COUNT(*)>1;

In [0]:
%sql
-- Null/invalid keys
SELECT 'sales_store_null' AS check_name,COUNT(*) AS issue_count FROM gold_fact_sales WHERE store_id IS NULL
UNION ALL SELECT 'sales_product_null',COUNT(*) FROM gold_fact_sales WHERE product_id IS NULL
UNION ALL SELECT 'inventory_store_null',COUNT(*) FROM gold_fact_inventory WHERE store_id IS NULL
UNION ALL SELECT 'inventory_product_null',COUNT(*) FROM gold_fact_inventory WHERE product_id IS NULL
UNION ALL SELECT 'po_supplier_null',COUNT(*) FROM gold_fact_purchase_orders WHERE supplier_id IS NULL;

In [0]:
%sql
-- Orphan store references
SELECT f.* FROM gold_fact_sales f
LEFT ANTI JOIN gold_dim_store d ON f.store_id=d.store_id;

In [0]:
%sql
-- Orphan product references
SELECT f.* FROM gold_fact_sales f
LEFT ANTI JOIN gold_dim_product d ON f.product_id=d.product_id;

In [0]:
%sql
-- Orphan supplier references
SELECT f.* FROM gold_fact_purchase_orders f
LEFT ANTI JOIN gold_dim_supplier d ON f.supplier_id=d.supplier_id;

In [0]:
%sql
-- Invalid prices
SELECT 'sales_invalid_price' AS check_name,COUNT(*) AS issue_count
FROM gold_fact_sales WHERE unit_price IS NULL OR unit_price<=0
UNION ALL
SELECT 'product_invalid_price',COUNT(*)
FROM gold_dim_product WHERE unit_price IS NULL OR unit_price<=0;

In [0]:
%sql
-- Invalid dates
SELECT 'sales_outside_2024' AS check_name,COUNT(*) AS issue_count
FROM gold_fact_sales
WHERE sale_date<'2024-01-01' OR sale_date>'2024-12-31'
UNION ALL
SELECT 'po_order_outside_2024',COUNT(*)
FROM gold_fact_purchase_orders
WHERE order_date<'2024-01-01' OR order_date>'2024-12-31'
UNION ALL
SELECT 'delivery_before_order',COUNT(*)
FROM gold_fact_purchase_orders
WHERE delivery_date IS NOT NULL AND delivery_date<order_date;

In [0]:
%sql
-- Negative quantities
SELECT 'negative_sales_quantity' AS check_name,COUNT(*) AS issue_count
FROM gold_fact_sales WHERE quantity_sold<0
UNION ALL
SELECT 'negative_inventory_quantity',COUNT(*)
FROM gold_fact_inventory WHERE quantity_on_hand<0;

In [0]:
%sql
-- Inventory quality
SELECT
    COUNT(*) AS inventory_rows,
    SUM(CASE WHEN quantity_on_hand=0 THEN 1 ELSE 0 END) AS out_of_stock_rows,
    SUM(CASE WHEN quantity_on_hand<50 THEN 1 ELSE 0 END) AS low_stock_rows,
    SUM(CASE WHEN quantity_on_hand<0 THEN 1 ELSE 0 END) AS negative_stock_rows
FROM gold_fact_inventory;

In [0]:
%sql
-- Purchase-order delivery-date quality
SELECT
    COUNT(*) AS purchase_order_rows,
    SUM(CASE WHEN delivery_date IS NULL THEN 1 ELSE 0 END) AS null_delivery_dates,
    SUM(CASE WHEN delivery_date IS NOT NULL AND delivery_date<order_date THEN 1 ELSE 0 END)
        AS delivery_before_order_rows
FROM gold_fact_purchase_orders;

In [0]:
%sql
-- Gold row counts
SELECT 'gold_dim_date' table_name,COUNT(*) row_count FROM gold_dim_date
UNION ALL SELECT 'gold_dim_store',COUNT(*) FROM gold_dim_store
UNION ALL SELECT 'gold_dim_product',COUNT(*) FROM gold_dim_product
UNION ALL SELECT 'gold_dim_supplier',COUNT(*) FROM gold_dim_supplier
UNION ALL SELECT 'gold_fact_sales',COUNT(*) FROM gold_fact_sales
UNION ALL SELECT 'gold_fact_inventory',COUNT(*) FROM gold_fact_inventory
UNION ALL SELECT 'gold_fact_purchase_orders',COUNT(*) FROM gold_fact_purchase_orders
UNION ALL SELECT 'gold_store_performance',COUNT(*) FROM gold_store_performance
UNION ALL SELECT 'gold_store_sales_monthly',COUNT(*) FROM gold_store_sales_monthly
UNION ALL SELECT 'gold_product_performance',COUNT(*) FROM gold_product_performance
UNION ALL SELECT 'gold_inventory_analysis',COUNT(*) FROM gold_inventory_analysis
UNION ALL SELECT 'gold_supplier_performance',COUNT(*) FROM gold_supplier_performance
UNION ALL SELECT 'gold_sales_monthly',COUNT(*) FROM gold_sales_monthly;

In [0]:
%sql
-- Revenue reconciliation
SELECT
    ROUND((SELECT SUM(total_amount) FROM gold_fact_sales),2) AS fact_sales_revenue,
    ROUND((SELECT SUM(total_revenue) FROM gold_store_performance),2) AS store_performance_revenue,
    ROUND(
        (SELECT SUM(total_amount) FROM gold_fact_sales)-
        (SELECT SUM(total_revenue) FROM gold_store_performance),2
    ) AS revenue_difference;

In [0]:
%sql
-- Sales totals reconciliation
SELECT
    (SELECT SUM(quantity_sold) FROM gold_fact_sales) AS fact_units_sold,
    (SELECT SUM(units_sold) FROM gold_store_performance) AS store_units_sold,
    (SELECT COUNT(DISTINCT transaction_id) FROM gold_fact_sales) AS fact_transactions,
    (SELECT SUM(transaction_count) FROM gold_store_performance) AS store_transactions;

In [0]:
%sql
-- Gold sanity checks
SELECT
    SUM(CASE WHEN total_revenue<0 THEN 1 ELSE 0 END) AS negative_revenue_rows,
    SUM(CASE WHEN transaction_count<0 THEN 1 ELSE 0 END) AS negative_transaction_count_rows,
    SUM(CASE WHEN market_share_percent<0 THEN 1 ELSE 0 END) AS negative_market_share_rows
FROM gold_store_performance;

## 12. Requirement-to-Gold Mapping

| Documented requirement | Gold source |
|---|---|
| Highest-revenue store | `gold_store_performance.total_revenue` |
| Top 5 products by quantity | `gold_product_performance.total_quantity_sold` |
| Store ranking by revenue | `gold_store_performance.revenue_rank` |
| Store ranking by transaction count | `gold_store_performance.transaction_count_rank` |
| Most profitable products | Not calculable as profit; revenue only |
| Month-on-month sales trend | `gold_store_sales_monthly` |
| Most diverse store | `gold_store_performance.unique_products_sold` |
| Out-of-stock | `gold_inventory_analysis.stock_status` |
| Low stock <50 | `gold_inventory_analysis.stock_status` |
| Highest sales velocity | `gold_product_performance.average_daily_sales_velocity` |
| Slow-moving | `gold_inventory_analysis.slow_moving_flag` |
| Overstock candidates | `gold_inventory_analysis.overstock_candidate_flag` |
| Average supplier delivery days | `gold_supplier_performance.average_delivery_days` |
| Supplier on-time rate | Not validly calculable |
| Supplier reliability | `gold_supplier_performance.supplier_reliability_proxy_rank` |
| Reorder point | `gold_inventory_analysis.reorder_point_proxy_units` |
| Total revenue by store | `gold_store_performance.total_revenue` |
| Average transaction value | `gold_store_performance.average_transaction_value` |
| Unique products sold | `gold_store_performance.unique_products_sold` |
| Inventory turnover | Proxy only |
| Top 3 categories | `gold_store_performance.top_3_categories_by_sales` |
| Market share | `gold_store_performance.market_share_percent` |
| Year/month trend | `gold_sales_monthly` / `gold_store_sales_monthly` |


## 13. Power BI Model / Relationships

Recommended one-to-many relationships:

- `gold_dim_date[date]` → `gold_fact_sales[sale_date]`
- `gold_dim_store[store_id]` → `gold_fact_sales[store_id]`
- `gold_dim_product[product_id]` → `gold_fact_sales[product_id]`
- `gold_dim_store[store_id]` → `gold_fact_inventory[store_id]`
- `gold_dim_product[product_id]` → `gold_fact_inventory[product_id]`
- `gold_dim_supplier[supplier_id]` → `gold_fact_purchase_orders[supplier_id]`

Use single-direction filtering from dimensions to facts.

For the dashboard pages, analytical marts are convenient dashboard-ready sources. Avoid unnecessary many-to-many relationships between marts and facts.


## 14. Dashboard 1 — Store Performance

**Primary:** `gold_store_performance`, `gold_store_sales_monthly`

Suggested visuals:
- Total revenue KPI
- Average transaction value KPI
- Transaction count KPI
- Unique products sold KPI
- Revenue by store
- Store revenue ranking
- Market share by store
- Monthly revenue trend by store
- Top 3 categories by store
- Store comparison table

Use the label **Inventory Turnover Proxy**.


## 15. Dashboard 2 — Product & Sales Analysis

**Primary:** `gold_product_performance`, `gold_store_performance`, `gold_sales_monthly`

Suggested visuals:
- Top 5 products by quantity
- Top products by revenue
- Category performance
- Store revenue ranking
- Store transaction ranking
- Monthly sales trend
- Product diversity by store
- Sales velocity

Do not create a profit visual because cost data is unavailable.


## 16. Dashboard 3 — Inventory & Supplier Analysis

**Primary:** `gold_inventory_analysis`, `gold_supplier_performance`

Suggested visuals:
- Out-of-stock products
- Low-stock products (<50)
- Fastest-moving products
- Slow-moving products
- Overstock candidates
- Average supplier delivery days
- Delivery completion rate
- Supplier reliability proxy rank
- Reorder-point proxy
- Inventory by store/product/category

Do not label delivery completion rate as on-time delivery rate.


## 17. Power BI Q&A Questions

### Store
- What is the total revenue by store?
- Which store has the highest total revenue?
- Show stores ranked by revenue.
- Show stores ranked by transaction count.
- What is the average transaction value by store?
- What is the market share percent by store?
- Show monthly revenue by store.
- Which store sold the most unique products?
- Show the top 3 categories by sales for each store.

### Product & Sales
- What are the top 5 products by quantity sold?
- What are the top products by total revenue?
- Show product revenue by category.
- Which products have the highest sales velocity?
- Show sales velocity by product.
- Which store has the most diverse product sales?
- Show monthly sales by store.

### Inventory & Supplier
- Which products are out of stock?
- Which products have low stock?
- Show products with stock below 50 units.
- Which products are the fastest moving?
- Which products are slow moving?
- Which products are overstock candidates?
- Show average delivery days by supplier.
- Which supplier has the highest delivery completion rate?
- Show supplier reliability proxy ranking.
- Show reorder point proxy units by store and product.
- Which products need reorder attention?

Do not use Q&A to ask for true profit or true on-time delivery because those fields are not supported by the source schema.


## 18. Refresh Strategy

Recommended sequence:

**Bronze/Raw → Silver validation → Silver cleaning → Gold → Power BI refresh**

Every Gold table uses `CREATE OR REPLACE TABLE`, so the notebook can be rerun after Silver is refreshed.

For production, these notebooks can later be orchestrated as a Databricks workflow.


## 19. Final Gold Table Inventory

### Dimensions
`gold_dim_date`, `gold_dim_store`, `gold_dim_product`, `gold_dim_supplier`

### Facts
`gold_fact_sales`, `gold_fact_inventory`, `gold_fact_purchase_orders`

### Analytical marts
`gold_store_performance`, `gold_store_sales_monthly`, `gold_product_performance`, `gold_inventory_analysis`, `gold_supplier_performance`, `gold_sales_monthly`

All are persistent Gold tables intended for Power BI.


In [0]:
%sql
select * from gold_store_performance

In [0]:
%sql
CREATE OR REPLACE TABLE gold_store_top3_category_sales AS

WITH category_sales AS (
    SELECT
        f.store_id,
        s.store_name,
        p.category,
        SUM(f.total_amount) AS category_sales
    FROM gold_fact_sales f
    JOIN gold_dim_store s
        ON f.store_id = s.store_id
    JOIN gold_dim_product p
        ON f.product_id = p.product_id
    GROUP BY
        f.store_id,
        s.store_name,
        p.category
),

ranked_categories AS (
    SELECT
        store_id,
        store_name,
        category,
        category_sales,
        ROW_NUMBER() OVER (
            PARTITION BY store_id
            ORDER BY category_sales DESC
        ) AS category_rank
    FROM category_sales
)

SELECT
    store_id,
    store_name,
    category,
    category_sales,
    category_rank
FROM ranked_categories
WHERE category_rank <= 3;

In [0]:
%sql
select store_name, category, category_sales from gold_store_top3_category_sales
sort by store_name, category_sales desc
